In [2]:
import pandas as pd
import os
import json
from data_generator import DataGenerator

In [20]:
if not os.path.exists("users.json"):
    dataGenerator = DataGenerator()
    dataGenerator.generate_users()
    
users_df = pd.read_json("users.json")

if not os.path.exists("products.json"):
    dataGenerator = DataGenerator()
    dataGenerator.generate_products()
    
products_df = pd.read_json("products.json")

if not os.path.exists("transactions.json"):
    dataGenerator = DataGenerator()
    dataGenerator.generate_transactions()
    
transactions_df = pd.read_json("transactions.json")


In [35]:
products_df = products_df.dropna()

removing invalid timestamps

In [36]:
ts_validity = pd.to_datetime(transactions_df["timestamp"], errors="coerce")
invalid_timestamps_transactions_df = transactions_df[ts_validity.isna()]
transactions_df_clean = transactions_df[ts_validity.notna()]
transactions_df_clean = transactions_df_clean.reset_index(drop=True)

C:\Users\NIYA\AppData\Local\Temp\ipykernel_13900\3070401251.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts_validity = pd.to_datetime(transactions_df["timestamp"], errors="coerce")


removing empty lists

In [37]:
transactions_df_clean = transactions_df_clean[transactions_df_clean["items"].apply(lambda x: isinstance(x, list) and len(x) > 0)]
transactions_df_clean = transactions_df_clean.reset_index(drop=True)

In [38]:
normalised_transactions_df = transactions_df_clean.explode("items")
normalised_items = pd.json_normalize(normalised_transactions_df["items"])
normalised_transactions_df = normalised_transactions_df.drop(columns=["items"]).reset_index(drop=True)
normalised_transactions_df = pd.concat([normalised_transactions_df, normalised_items], axis=1)

tuk e topa

In [39]:
normalised_transactions_df

,transaction_id,user_id,timestamp,status,product_id,quantity
0,1,3,2024-08-07T12:44:37,RETURNED,0,15
1,1,3,2024-08-07T12:44:37,RETURNED,19,14
2,2,4,2024-10-18T00:44:54,RETURNED,4,13
3,2,4,2024-10-18T00:44:54,RETURNED,14,14
4,2,4,2024-10-18T00:44:54,RETURNED,8,11
...,...,...,...,...,...,...
65,27,3,2023-04-08T12:33:10,COMPLETED,15,5
66,27,3,2023-04-08T12:33:10,COMPLETED,19,4
67,29,6,2024-09-12T00:47:40,COMPLETED,4,14
68,29,6,2024-09-12T00:47:40,COMPLETED,15,5


In [40]:
merged_transactions_users_df = normalised_transactions_df.merge(users_df, on="user_id")

In [42]:
merged_full_df = merged_transactions_users_df.merge(products_df, on="product_id")

In [45]:
merged_full_df["total_item_value"] = merged_full_df["price"] * merged_full_df["quantity"]
merged_full_df = merged_full_df[["transaction_id", "timestamp", "status", "product_id", "category", "quantity", "price", "total_item_value", "user_id", "name", "country"]]

In [48]:
merged_full_df.to_csv("cleaned_dataset.csv", index=False)